In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.filtering import importance_sampling
from pism_terra.processing import preprocess_netcdf as preprocess

data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {'surface.debm_simple.c1': 'c1', 
                'surface.debm_simple.c2': 'c2',
                'surface.debm_simple.air_temp_all_precip_as_snow': 'as_snow', 
                'surface.debm_simple.air_temp_all_precip_as_rain': 'as_rain', 
                'surface.debm_simple.refreeze': 'refreeze'}

pdd_uq_vars = {'surface.pdd.factor_ice': 'fice', 
               'surface.pdd.factor_snow': 'fsnow',
               'surface.pdd.refreeze': 'refreeze'}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc", engine="netcdf4",
                     decode_times=False,
                     decode_timedelta=False, chunks=None).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1000]):
    
    ds = xr.open_mfdataset(f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
                         preprocess=partial(preprocess,
                                            uq_regexp=None,
                                            exp_regexp="uq_(.+?)_"),
                        engine="netcdf4",
                        join="outer",
                        compat="no_conflicts",
                        parallel=True,
                        chunks="auto",
                        decode_times=False,
                        decode_timedelta=False).drop_dims("nv", errors="ignore")

    
    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()                                                                                                                                         
    _obs[m_vars] = _obs[m_vars].where(mask) 
    melt_mask = (_obs["climatic_mass_balance"].mean(dim="time") < 0)
    _obs[m_vars] = _obs[m_vars].where(melt_mask) 
    _ds = ds[m_vars].where(melt_mask)
    
    for v in m_vars:
    
        with ProgressBar():
            ebm_filtered = importance_sampling(_ds, _obs, 
                            sim_var=v,
                            obs_mean_var=v, 
                            obs_std_var=f"{v}_error",
                            sum_dims=['time', 'x', 'y'],
                            n_samples=ds.exp_id.size,
                            fudge_factor=fudge_factor
                                              )

            ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
            ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
                                                                                                                                                                                                 
            # Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
            ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)                                                                                                                
            most_sampled_id = ebm_counts.idxmax()                                                                                                                                  
            most_sampled_params = ebm_uq_df.loc[most_sampled_id]
            print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")                                                                                 
            for k, short in ebm_uq_vars.items():                                                                                                                                   
                print(f"  {short}: {most_sampled_params[k]:.6g}")     
                  
            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))           
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):                                                                                                                                             
                    # Repeat each parameter value by its sample count                                                                                                                              
                      values = np.repeat(ebm_uq_df[key].values,                                                                                                                                      
                                     ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
                      ax.hist(values, bins=15)                                                                                                                                                       
                      ax.set_xlabel(value)    
                      ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                      # ax.set_ylabel("Count")                                                                                                                                                         
            fig.tight_layout()   
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig
            
            
            # Plot smallest RMSE                                                                                                                                                       
            rmse = xs.rmse(_ds[v].mean(dim="time"), _obs[v].mean(dim="time"), dim=["x", "y"], skipna=True).compute()
            best_id = rmse.idxmin(dim="exp_id").values                                                                                                                                                                                                                                                                                                                          
            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()                                                                                                            
            obs_mean = _obs[v].mean(dim="time").squeeze()                                                                                                                               
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]                                                                                                                                                                                         
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))                                                                                                                         
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)                                                                                                                            
            axes[0].set_title("Observed")                                                                                                                                              
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)                                                                                                                            
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())                                                                                                 
            axes[1].set_title(f"Best (id={best_id}, RMSE={float(rmse.sel(exp_id=best_id)):.1f})\n{param_str}") 
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-2000, vmax=2000)                                                                                                                        
            axes[2].set_title("Difference")                   
            fig.tight_layout()                                                                                                                                                         
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)                                                                                                                           
            plt.close()
            del fig 

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.processing import preprocess_netcdf as preprocess


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1000]):

    ds = xr.open_mfdataset(
        f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
        preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
        engine="netcdf4",
        join="outer",
        compat="no_conflicts",
        parallel=True,
        chunks="auto",
        decode_times=False,
        decode_timedelta=False,
    ).drop_dims("nv", errors="ignore")

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 0
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in m_vars:

        with ProgressBar():

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = pd.DataFrame(
                {
                    "rmse_mean": rmse_mean.values,
                    "rmse_lo": rmse_lo.values,
                    "rmse_hi": rmse_hi.values,
                    "tied_with_best": tied_mask.values,
                },
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            ).join(ebm_uq_df, how="left").sort_values("rmse_mean")
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-2000, vmax=2000)
            axes[2].set_title("Difference")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
_obs["climatic_mass_balance"].mean(dim="time").plot()

In [ ]:
sampled_ids = debm_filtered.exp_id_sampled.values
counts = pd.Series(sampled_ids).value_counts()
                                                                                                                                                                                     
# Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
sampled_configs = debm_uq_df.loc[counts.index].reindex(counts.index)                                                                                                                
                                                                                                                                                                                     
fig, axes = plt.subplots(2, 3, figsize=(12, 4))           
for ax, var in zip(axes.flat, debm_uq_vars):                                                                                                                                             
        # Repeat each parameter value by its sample count                                                                                                                              
          values = np.repeat(debm_uq_df[var].values,                                                                                                                                      
                         counts.reindex(debm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
          ax.hist(values, bins=15)                                                                                                                                                       
          ax.set_xlabel(var)                                    
          ax.set_ylabel("Count")                                                                                                                                                         
plt.tight_layout()          

In [ ]:
_debm_filtered.exp_id_sampled

In [ ]:
df = pdd.pism_config.to_series().apply(json.loads).apply(pd.Series)[pdd_uq_vars]

In [ ]:
import pandas as pd

In [ ]:
            rmse = xs.rmse(_ds[v].mean(dim="time"), _obs[v].mean(dim="time"), dim=["x", "y"], skipna=True).compute()
            best_id = rmse.idxmin(dim="exp_id").values   

In [ ]:
best_id

In [ ]:
ds = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = ds_scalar.resample(time="YE").mean()

ds_new = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm_strong/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_new = ds_new.sel(basin="GIS")
#ds_new = ds_new.resample(time="YE").mean()


fig, ax = plt.subplots(1, 1)
#ds.sel(basin="GIS").tendency_of_ice_mass.plot(ax=ax)
#ds_scalar.tendency_of_ice_mass.plot(ax=ax)
ds_new.tendency_of_ice_mass.plot(ax=ax)



In [ ]:
ds_scalar.tendency_of_ice_mass.plot(ax=ax)

In [ ]:
ds_scalar.tendency_of_ice_mass.plot()

In [ ]:
ds_sampled_configs

In [ ]:
ds = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm_mode/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0005-01-01.nc")
ds.sel(basin="GIS").tendency_of_ice_mass.plot()

In [ ]:
sds = xr.open_dataset("../fldsum_spatial_g2400m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")

In [ ]:
sds.tendency_of_ice_mass.mean(dim="time")